# MSADS 599 - Modeling Part 1 - Classification - Random Forest

## Setup

Import Packages

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download, login
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder, StandardScaler

# This shows all columns when analyzing the data
pd.set_option('display.max_columns', None)

Connect to HuggingFace and Import Data

In [ ]:
# Connection to HuggingFace via API token
login(token="HF_TOKEN")

In [ ]:
# Import training data
train_path = hf_hub_download(
    repo_id = "ADS599-Capstone/modeling_data",
    filename = "part1_train/part1_train-00000-of-00001.parquet",
    repo_type = "dataset"
)

train_df = pd.read_parquet(train_path)
print(train_df.shape)

In [ ]:
# Import testing data
test_path = hf_hub_download(
    repo_id = "ADS599-Capstone/modeling_data",
    filename = "part1_test/part1_test-00000-of-00001.parquet",
    repo_type = "dataset"
)

test_df = pd.read_parquet(test_path)
print(test_df.shape)

## Modeling

Clean and Set the Target Variable

In [ ]:
# Encode the target variable
le = LabelEncoder()
train_df['target'] = le.fit_transform(train_df['cohort_label'])
test_df['target'] = le.fit_transform(test_df['cohort_label'])

# Check the mapping of the class to the number
class_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(class_mapping)

In [ ]:
# Split the target variable from the test and train datasets

# Training Set
X_train = train_df.drop(columns=['cohort_label', 'target'])
y_train = train_df['target']

# Testing Set
X_test = test_df.drop(columns=['cohort_label', 'target'])
y_test = test_df['target']

# Sanity Check
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

Train the Model

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight='balanced',
    random_state=35,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

Evaluate the Model

In [ ]:
y_pred = rf_model.predict(X_test)

print(classification_report(y_test, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(xticks_rotation=45)
plt.tight_layout()
plt.show()

Feature Importance - Random Forest

In [ ]:
importances = rf_model.feature_importances_
feature_names = X_train.columns
indices = np.argsort(importances)[::-1][:20] # Top 20 features

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(20), importances[indices][::-1], color='steelblue')
ax.set_yticks(range(20))
ax.set_yticklabels(feature_names[indices][::-1])
ax.set_title("Top 20 Most Importance Features")
ax.set_xlabel("Feature Importance (GINI Impurity)")
plt.tight_layout()
plt.show()